In [1]:
import numpy as np
import numpy.typing as npt
import qutip as qt
from qiskit.circuit import ClassicalRegister, Gate, QuantumCircuit, QuantumRegister
from qiskit.circuit.library import StatePreparation
from qiskit.quantum_info import Operator, Statevector, partial_trace
from qiskit_encore.initializer import kernel_based_initializer_operator
from qiskit_encore.preparable_statevector import (
    BigUnitaryPreparableStatevector,
    KernelBasedPreparableStatevector,
    PreparableStatevector,
)
from qiskit_signals.sample_based_signal import ArbitrarySignalForSampleBasedProtocol


In [2]:
def slice_alpha_to_deltas_evenly(alpha: float, max_delta: float) -> npt.NDArray:
    """Slices the alpha value into a list of deltas, each with a maximum value of max_delta.

    Args:
        alpha (float): The total alpha value to be sliced.
        max_delta (float): The maximum value for each delta slice.

    Returns:
        npt.NDArray: An array of delta values that sum up to alpha.
    """

    # assure delta is positive
    if max_delta <= 0:
        raise ValueError("The max_delta must be positive.")

    max_delta *= np.sign(alpha)

    number_of_deltas = int(np.ceil(np.abs(alpha / max_delta)))
    delta = alpha / number_of_deltas
    deltas = delta * np.ones(number_of_deltas)

    return deltas


def householder_unitary(psi: Statevector) -> qt.Qobj:
    """Construct a unitary U such that U|0> = psi using a Householder reflection."""
    N = psi.dim
    ket0 = qt.basis(N, 0)

    qt_state = qt.Qobj(psi.data)

    # Check if target is already |0>
    if (ket0 - qt_state).norm() < 1e-12:
        return qt.identity(N)

    c = ket0.dag() @ qt_state  # <0|psi>
    corrector_phase = np.exp(-1j * np.angle(c))

    psi_phase_corrected = corrector_phase * qt_state

    # Compute the Householder vector
    v = (ket0 - psi_phase_corrected).unit()  # normalized
    U = qt.identity(N) - 2 * v * v.dag()

    U_corrected_global_phase = np.exp(1j * np.angle(c)) * U

    return U_corrected_global_phase

In [3]:
import matplotlib.pyplot as plt
import scipy.constants as constants
from qiskit import transpile
from qiskit.quantum_info import Statevector

# from qiskit_aer_encore.simulator import generate_aer_simulator
from qiskit_hamiltonian_simulation.time_independent.direct import (
    MomentumDomainEvolutionQuadratic,
    PositionDomainEvolutionQuadratic,
)
from qiskit_hamiltonian_simulation.time_independent.sample_based import (
    KineticEvolutionSampleBased,
    PotentialEvolutionSampleBased,
)
from qiskit_phase_propagator.sample_based_manual import (
    phase_propagate_state_with_arbitrary_signal,
)

# from qiskit_phase_propagator.sample_based import (
#     QuadraticSignalSampleBasedPhasePropagator,
# )
from qiskit_signals.helper_types import EncodingType
from qiskit_signals.quantum_axis import (
    AngularWavenumberAxis,
    MomentumAxis,
    PositionAxis,
)
from qiskit_signals.quantum_signal import GenericQuantumSignal, QuadraticQuantumSignal
from wave_optics_propagation.analytics import (
    free_space_propagated_gaussian_wavefront,
    gaussian_signal,
    propagated_gaussian_wavefront_hitting_lens,
)
from wave_optics_propagation.big_matrix_version import big_matrix_circ
from wave_optics_propagation.elements import (
    free_space_signal_generator,
    radius_of_convex_planar_lens_as_a_func_of_z,
    thin_lens_signal_generator,
    thin_transparent_plate_signal_generator,
)
from wave_optics_propagation.storage import (
    save_initial_parameters,
    save_numpy_results,
)
from wave_optics_propagation.visualization import plot_wavefunction

vacuum_wavelength = 1e-6 * 1e1
# beam_FWHM = 5e-3 * 1e-2
beam_FWHM = 20e-3 * 1e-2

# lens parameters
focal_length = 200e-3 * 1e-2
# lens_diameter = 25e-3
# refractive_index = 1.5
refractive_index = 1.25

# free space propagation parameters
propagation_after_lens = 1.5 * focal_length

# simulation parameters
transverse_length = 100e-3 * 1e-2  # transverse simulation window
num_of_steps_after_lens = 10
lens_slices = 10000
num_qubits = 2
max_delta = 0.025

# lens_diameter = 50e-3 * 1e-2
lens_diameter = transverse_length
### Constants

c = constants.c
# hbar = constants.hbar
### Derived parameters

# geometry
radius_of_curvature = focal_length * (refractive_index - 1)
lens_radius = lens_diameter / 2
lens_thickness = radius_of_curvature - np.sqrt(radius_of_curvature**2 - lens_radius**2)


k_0 = 2 * constants.pi / vacuum_wavelength
reduced_wavelength = vacuum_wavelength / refractive_index

dimension = 2**num_qubits
delta_x = transverse_length / dimension

gaussian_mean = transverse_length / 2
gaussian_beam_waist = beam_FWHM / np.sqrt(2 * np.log(2))
# gaussian_sigma = gaussian_beam_waist / np.sqrt(2)

# vacuum_rayleigh_length = (np.pi * gaussian_beam_waist**2) / vacuum_wavelength

lens_slice_thickness = lens_thickness / lens_slices
step_size_after_lens = propagation_after_lens / num_of_steps_after_lens
print(delta_x, vacuum_wavelength)
### Derived objects

lens_slice_positions = (
    np.linspace(0, lens_thickness, lens_slices, endpoint=False)
    + lens_slice_thickness / 2
)  # choosing the midpoint in each transverse slice

lens_transverse_radii = [
    radius_of_convex_planar_lens_as_a_func_of_z(
        radius_of_curvature, z, lens_thickness, fresnel_approximation=True
    )
    for z in lens_slice_positions
]
print(lens_thickness)
print(lens_slice_positions)
print(lens_transverse_radii)
### Approximations checks

assert (
    gaussian_beam_waist > 10 * vacuum_wavelength
)  # ensure paraxial approximation validity
### Axes

x_axis = PositionAxis(
    num_qubits=num_qubits, delta_x=delta_x, encoding=EncodingType.UNSIGNED
)
k_axis = AngularWavenumberAxis.from_position_axis(x_axis)
# p_axis = MomentumAxis.from_position_axis(x_axis, hbar=hbar)
### Initial beam profile

# def psi_signal_function(x):
#     norm_factor = 1 / (gaussian_sigma * (2 * constants.pi) ** 0.25)
#     return norm_factor * np.exp(-(((x - gaussian_mean) / gaussian_beam_waist) ** 2))


# def psi_signal_function(x):
#     return np.where(
#         (x > 1 / 3) & (x < 2 / 3),
#         1,
#         0,
#     )


# def psi_signal_function(x):
#     signal = np.zeros(dimension)
#     middle = dimension // 2
#     signal[middle - 20 : middle - 15] = 1
#     signal[middle + 15 : middle + 20] = 1
#     return signal

# psi_flat = Statevector.from_label("+" * num_qubits)

# psi = GenericQuantumSignal(x_axis, psi_signal_function)

psi = gaussian_signal(x_axis, gaussian_beam_waist, gaussian_mean)

# print(np.linalg.norm(psi.data) ** 2)
print(gaussian_beam_waist, beam_FWHM, gaussian_mean)
# plot_wavefunction(psi.data, normalize=False)

### Lens potentials

lens_signals = [
    thin_transparent_plate_signal_generator(
        x_axis=x_axis,
        refractive_index=refractive_index,
        thickness=lens_slice_thickness,
        radius=lens_radius,
        wavelength=vacuum_wavelength,
        scale_down=True,
    )
    for lens_radius in lens_transverse_radii
]
NUM_OF_PROFILES_TO_PLOT = 5

# for i in range(NUM_OF_PROFILES_TO_PLOT):
#     plot_wavefunction(lens_signals[i].data, normalize=False)
# plot_wavefunction(lens_signals[-1].data, normalize=False)
sum_signal = np.zeros_like(lens_signals[0].data)
for signal in lens_signals:
    sum_signal += signal.data


0.00025 9.999999999999999e-06
0.0005
[2.50000e-08 7.50000e-08 1.25000e-07 ... 4.99875e-04 4.99925e-04
 4.99975e-04]
[np.float64(5e-06), np.float64(8.660254037844387e-06), np.float64(1.1180339887498949e-05), np.float64(1.3228756555322953e-05), np.float64(1.5e-05), np.float64(1.6583123951777e-05), np.float64(1.8027756377319944e-05), np.float64(1.9364916731037083e-05), np.float64(2.06155281280883e-05), np.float64(2.179449471770337e-05), np.float64(2.2912878474779198e-05), np.float64(2.3979157616563598e-05), np.float64(2.4999999999999998e-05), np.float64(2.598076211353316e-05), np.float64(2.692582403567252e-05), np.float64(2.783882181415011e-05), np.float64(2.8722813232690143e-05), np.float64(2.958039891549808e-05), np.float64(3.0413812651491098e-05), np.float64(3.1224989991991994e-05), np.float64(3.201562118716424e-05), np.float64(3.278719262151e-05), np.float64(3.3541019662496847e-05), np.float64(3.427827300200522e-05), np.float64(3.5e-05), np.float64(3.570714214271425e-05), np.float64(3

In [4]:
from wave_optics_propagation.analytics import gaussian_signal

psi_in: qt.Qobj = qt.Qobj(psi.data)
signal: ArbitrarySignalForSampleBasedProtocol = (
    ArbitrarySignalForSampleBasedProtocol.from_generic_signal(lens_signals[7])
)
max_delta: float = 0.1

# calculate the deltas from the signal and the corresponding statevector
alpha, state = signal.alpha, signal.statevector
deltas = slice_alpha_to_deltas_evenly(alpha, max_delta)

# create the initializer for the state
U_phi = householder_unitary(state)
U_phi

Quantum object: dims=[[4], [4]], shape=(4, 4), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]]

In [5]:
state.draw()

'Statevector([0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j],\n            dims=(2, 2))'

In [6]:
U_phi_dagger = U_phi.dag()

# define tools
N = state.dim
delta = deltas[0]  # deltas are all the same here
ZERO_STATE = qt.basis(N, 0)
num_cycles = len(deltas)
output_state = psi_in.copy()

output_state.full()


array([[1.72633492e-04+0.j],
       [1.14625505e-01+0.j],
       [1.00000000e+00+0.j],
       [1.14625505e-01+0.j]])

In [7]:
# input_state = psi_in.tensor(ZERO_STATE)
input_state = qt.tensor(ZERO_STATE, psi_in)

input_state.full()

array([[1.72633492e-04+0.j],
       [1.14625505e-01+0.j],
       [1.00000000e+00+0.j],
       [1.14625505e-01+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j]])

In [8]:
# applying phi initializer
operator = qt.tensor(U_phi, qt.qeye(N))
operator

Quantum object: dims=[[4, 4], [4, 4]], shape=(16, 16), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]

In [9]:
psi_before_phase_unit = operator @ input_state

psi_before_phase_unit.full()

array([[0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [1.72633492e-04+0.j],
       [1.14625505e-01+0.j],
       [1.00000000e+00+0.j],
       [1.14625505e-01+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j]])

In [10]:
# Evolve the input state through the circuit
# TODO: this might be the tricky part, but the operator I think was actually just diagonal so we do not really need to build the full circuit here
# TODO: let us not even decompose to qubits anymore.
operator = qt.tensor(qt.qzero(N), qt.qeye(N))
for j in range(N):
    diagonals = np.ones(N)
    diagonals[j] = np.exp(1j * delta)
    operator += qt.tensor(qt.basis(N, j).proj(), qt.qdiags(diagonals, 0))

psi_after_phase_unit = operator @ psi_before_phase_unit

psi_after_phase_unit.full()


/tmp/ipykernel_2071453/1229463502.py:7: ComplexWarning: Casting complex values to real discards the imaginary part
  diagonals[j] = np.exp(1j * delta)


array([[0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [1.72633492e-04+0.j],
       [1.14625505e-01+0.j],
       [9.99969158e-01+0.j],
       [1.14625505e-01+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j]])

In [14]:
# applying phi de-initializer
operator = qt.tensor(U_phi_dagger, qt.qeye(N))
psi_out_pre_projection = operator @ psi_after_phase_unit

psi_out_pre_projection.full()

array([[1.72633492e-04+0.j],
       [1.14625505e-01+0.j],
       [9.99969158e-01+0.j],
       [1.14625505e-01+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j]])

In [16]:
zero_state_projector = ZERO_STATE.proj()
operator = qt.tensor(zero_state_projector, qt.qeye(N))
psi_out_post_projection = operator @ psi_out_pre_projection

psi_out_post_projection.full()

array([[1.72633492e-04+0.j],
       [1.14625505e-01+0.j],
       [9.99969158e-01+0.j],
       [1.14625505e-01+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j],
       [0.00000000e+00+0.j]])

In [17]:
psi_out_traced_phi = psi_out_post_projection.proj().ptrace(0)
psi_out_traced_phi.full()

array([[1.02621636+0.j, 0.        +0.j, 0.        +0.j, 0.        +0.j],
       [0.        +0.j, 0.        +0.j, 0.        +0.j, 0.        +0.j],
       [0.        +0.j, 0.        +0.j, 0.        +0.j, 0.        +0.j],
       [0.        +0.j, 0.        +0.j, 0.        +0.j, 0.        +0.j]])

In [12]:
# TODO: just directly extract the relevant part of the statevector instead of doing all this projection and tracing out

# Post-select on the |0...0> outcome of the phi register measurement
# zero_state_projector = ZERO_STATE.proj()
# operator = qt.tensor(zero_state_projector, qt.qeye(N))
# psi_out_post_projection = operator @ psi_out_pre_projection

# # trace out the phi register
# print(psi_out_post_projection.dims)
# test = psi_out_post_projection[0]
# print(test.dims)
# psi_out_traced_phi = psi_out_post_projection.proj().ptrace(0)
# print(psi_out_traced_phi.dims)
state_array = psi_out_pre_projection[0:N]
projected_state = qt.Qobj(state_array)

projected_state


Quantum object: dims=[[4], [1]], shape=(4, 1), type='ket', dtype=Dense
Qobj data =
[[1.72633492e-04]
 [1.14625505e-01]
 [9.99969158e-01]
 [1.14625505e-01]]

In [18]:
# renormalize
normalized_psi_out_post_projection = projected_state.unit()

normalized_psi_out_post_projection.full()


array([[1.70414124e-04+0.j],
       [1.13151886e-01+0.j],
       [9.87113607e-01+0.j],
       [1.13151886e-01+0.j]])